In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/superheated_vapor.csv')
df.head()

,Pressure,Property,Liq_Sat,Vap_Sat,75,100,125,150,175,200,...,425,450,475,500,525,550,575,600,625,650
0,1.0,V,1.000,129200.0000,160640.0000,172180.0000,183720.0000,195270.0000,206810.0000,218350.0000,...,NaN,333730.00,NaN,356810.0000,NaN,379880.0000,NaN,402960.0000,NaN,426040.0000
1,1.0,U,29.334,2385.2000,2480.8000,2516.4000,2552.3000,2588.5000,2624.9000,2661.7000,...,NaN,3049.90,NaN,3132.4000,NaN,3216.7000,NaN,3302.6000,NaN,3390.3000
2,1.0,H,29.335,2514.4000,2641.5000,2688.6000,2736.0000,2783.7000,2831.7000,2880.1000,...,NaN,3383.60,NaN,3489.2000,NaN,3596.5000,NaN,3705.6000,NaN,3816.4000
3,1.0,S,0.106,8.9767,9.3828,9.5136,9.6365,9.7527,9.8629,9.9679,...,NaN,10.82,NaN,10.9612,NaN,11.0957,NaN,11.2243,NaN,11.3476
4,10.0,V,1.010,14670.0000,16030.0000,17190.0000,18350.0000,19510.0000,20660.0000,21820.0000,...,NaN,33370.00,NaN,35670.0000,NaN,37980.0000,NaN,40290.0000,NaN,42600.0000


In [4]:
# Multiple Regression , Enthalpy H with Pressure, P and Temperature, T

H = df.loc[df["Property"] == "H"]
H. shape


(136, 37)

In [5]:
H.head()

,Pressure,Property,Liq_Sat,Vap_Sat,75,100,125,150,175,200,...,425,450,475,500,525,550,575,600,625,650
2,1.0,H,29.335,2514.4,2641.5,2688.6,2736.0,2783.7,2831.7,2880.1,...,NaN,3383.6,NaN,3489.2,NaN,3596.5,NaN,3705.6,NaN,3816.4
6,10.0,H,191.832,2584.8,2640.0,2687.5,2735.2,2783.1,2831.2,2879.6,...,NaN,3383.5,NaN,3489.1,NaN,3596.5,NaN,3705.5,NaN,3816.3
10,20.0,H,251.453,2609.9,2638.4,2686.3,2734.2,2782.3,2830.6,2879.2,...,NaN,3383.4,NaN,3489.0,NaN,3596.4,NaN,3705.4,NaN,3816.2
14,30.0,H,289.302,2625.4,2636.8,2685.1,2733.3,2781.6,2830.0,2878.7,...,NaN,3383.3,NaN,3488.9,NaN,3596.3,NaN,3705.4,NaN,3816.2
18,40.0,H,317.650,2636.9,NaN,2683.8,2732.3,2780.9,2829.5,2878.2,...,NaN,3383.1,NaN,3488.8,NaN,3596.2,NaN,3705.3,NaN,3816.1


In [7]:
# There will be two input P and T and one output H.

X = H["Pressure"]
y = y = np.array([int(H.columns[4:][i]) for i in range(len(H.columns[4:]))])
Z = Z = H.loc[:, '75':'650']

In [8]:
y

array([ 75, 100, 125, 150, 175, 200, 220, 225, 240, 250, 260, 275, 280,
       290, 300, 320, 325, 340, 350, 360, 375, 380, 400, 425, 450, 475,
       500, 525, 550, 575, 600, 625, 650])

In [13]:
# Flatten the data
P_values = H['Pressure'].to_numpy()                              # (136,)
T_values = H.columns[4:].astype(int).to_numpy()                  # (33,)

# Turn it into a meshgrid 
P_grid, T_grid = np.meshgrid(P_values, T_values)                 # both (33, 136)
H_grid = H.loc[:, '75':'650'].to_numpy().T   
                    # (33, 136)
# Flatten 
P_flat = P_grid.ravel()      # (4488,)
T_flat = T_grid.ravel()      # (4488,)
H_flat = H_grid.ravel()      # (4488,)

# Remove NaN values
mask = ~np.isnan(H_flat)
P_clean = P_flat[mask]
T_clean = T_flat[mask]
H_clean = H_flat[mask]

print(P_flat.shape, mask.sum(), P_clean.shape)   # (4488,)  2063  (2063,)

(4488,) 2063 (2063,)


In [20]:
# Fit the multilinear Regression
import sys
sys.path.append("..") 
from functions import polynomial_model
from sklearn.metrics import mean_squared_error

# Fit the model
X_fit = np.column_stack((P_clean, T_clean))

for i in range(1, 4):
    model = polynomial_model(P_clean, T_clean, H_clean, i)
    lr = model.named_steps["linear_regression"]
    names = model.named_steps["polynomial_features"].get_feature_names_out(["P", "T"])
    y_pred = model.predict(X_fit)
    mse = mean_squared_error(H_clean, y_pred)
    rmse = np.sqrt(mse)

    print(f"Polynomial of degree {i}")
    print("R2 score:", model.score(X_fit, H_clean))
    print("MSE:", mse)
    for name, c in zip(names, lr.coef_):
        print(f"  {name}: {c:.4f}")
    print("Intercept:", lr.intercept_)
    print()


Polynomial of degree 1
R2 score: 0.9876784909472981
MSE: 1128.774185114659
  P: -58.2739
  T: 313.3289
Intercept: 3249.0192922927804

Polynomial of degree 2
R2 score: 0.9978173319626839
MSE: 199.95435012540506
  P: -68.1215
  T: 324.6884
  P^2: -0.0164
  P T: 35.8999
  T^2: -3.8265
Intercept: 3241.960347958279

Polynomial of degree 3
R2 score: 0.9995744023099568
MSE: 38.989029972741065
  P: -53.7137
  T: 327.7685
  P^2: -4.2741
  P T: 49.6892
  T^2: -15.5919
  P^3: 0.0245
  P^2 T: 3.1116
  P T^2: -21.2536
  T^3: 3.6911
Intercept: 3248.340373236339



In [25]:
# Adding in log features
import sys
sys.path.append("..") 
from features import log_feature_PT
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# X_fit = np.column_stack((P_clean, T_clean))
model_Hlog = Pipeline([
    ("log_basis", log_feature_PT()),
    ("scaler", StandardScaler()),
    ("linear_regression", LinearRegression()),
]).fit(X_fit, H_clean)

y_pred = model_Hlog.predict(X_fit)
print("R2:", model_Hlog.score(X_fit, H_clean))
print("RMSE:", np.sqrt(mean_squared_error(H_clean, y_pred)))
print(model_Hlog.named_steps["linear_regression"].coef_)

R2: 0.9941573213802574
RMSE: 23.135430396461317
[-61.80669685 222.04579263 -62.31898446 127.74451161]


In [26]:
# printing the coefficients
def print_equation(model, names):
    lr = model.named_steps["linear_regression"]
    sc = model.named_steps["scaler"]
    a = lr.coef_ / sc.scale_
    a0 = lr.intercept_ - np.sum(lr.coef_ * sc.mean_ / sc.scale_)
    print(f"a0: {a0:.4f}")
    for n, c in zip(names, a):
        print(f"{n}: {c:.6g}")

print_equation(model_Hlog, ["P", "T", "ln P", "T*ln P"])

a0: 2641.1620
P: -0.0191933
T: 1.60687
ln P: -38.7794
T*ln P: 0.0969206
